# Random Forest for RQ4: Can bagging make detection on an unseen day more stable ?

### Why Use Random Forest Model ?

We are adding a Random Forest model because it helps us support our predictive research question:

RQ4: Using leave-one-day-out evaluation on deduplicated data, how well does a benign-vs-malicious model detect the held-out day's attacks, all of which are unseen types, and how much of any drop comes from shifted benign traffic rather than from the new attacks?

Right now our best model is the histogram gradient boosting (HGB). HGB is a boosting ensemble, where it attempts to fix mistakes of trees created before.

When we reran HGB with different random seeds `m3_stability.csv`, most days barely changed. Tuesday did not. With the same data, features, and settings, Tuesday's recall jumped from 1.9% to 22% just from changing the seed. This situation can be considered high variance since the model's results change a lot when only the random seed changes, so a single run does not give a reliable estimate of how well the model detects Tuesday's attacks.

A common way to reduce the variance is bagging, and a Random Forest is a form of bagging with decision trees. Each tree trains independently on its own random sample of rows and only looks at a random handful of features at each split. Averaging many different trees should cancel out each tree's random mistakes.

Overall this notebook tests whether a Random Forest is more stable than HGB on an unseen day, and does it detect attacks as well ? 

To keep the comparison fair, the forest uses the exact same setup as the RQ4 models: leave-one-day-out, the same 64 features, the same 1% false alarm threshold set from benign training traffic, and the same metrics.

### Loading the necessary libraries and shared tools from `m3_common.py`

Using the team's shared tools means the Random Forest follows the same rules as logistic regression and HGB.

In [2]:
# Import Libraries
import sys
from pathlib import Path
import numpy as np 
import pandas as pd

# Setting Data And Output folders
DATA_DIR = Path("../data/processed")
OUT_DIR = Path("../results/m3")

# Add src folder to path
sys.path.insert(0, "../src")

# Import tools from m3_common
from m3_common import evaluate, feature_list, load_clean, pick_threshold, split_fit_calib

### Afterwards we load the cleaned data and the 64 model features 

In [33]:
# Load Cleaned Data 
df = load_clean(DATA_DIR)

# Getting the 64 features
FEATS = feature_list(df)

# Setting up features, target, labels, and days from RQ4
X = df[FEATS].to_numpy(np.float32)
y = df["is_attack"].to_numpy()
labels = df["Label"].to_numpy()
day = df["day"].to_numpy()

# Free Up Memory
del df

# Checking Shape 
print("The shape of X:", X.shape)

The shape of X: (2520656, 64)


### Random Forest

Predictions of multiple decision trees.

Algorithm Steps 
1. Bootstrap Sampling: Each tree trains on its own random sample of rows.
2. Random Feature selection: Each split only considers a random subset of features.
3. Voting: The forest averages the trees' votes into an attack probability.

In [5]:
# import Random Forest 
from sklearn.ensemble import RandomForestClassifier

### Hold out one day

Instead of a random train/test split, we train on four days and test on the fifth, so every attack in the test day is a type the model has never seen. We start with Tuesday, the day where HGB was unstable.

In [6]:
# Choose the held-out day "Tuesday"
test_day = "Tuesday"

# Get row numbers for training days and test day
train_idx = np.flatnonzero(day != test_day)
test_idx = np.flatnonzero(day == test_day)

# Check sizes
print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))

Training rows: 2099107
Test rows: 421549


### Splitting the training days into fit and calibration

The training rows are split 80/20 with the team's `split_fit_calib`, the same split that was used for the HGB model. The forest learns from the fit rows. The calibration rows are used to set the alert threshold, so the threshold comes from data the model did not train on.

In [7]:
# Split training rows into fit and calibration 
fit_idx, cal_idx = split_fit_calib(train_idx, labels)

# Setting up train, calibration and test data 
X_train = X[fit_idx]
y_train = y[fit_idx]
X_cal = X[cal_idx]
y_cal = y[cal_idx]
X_test = X[test_idx]
y_test = y[test_idx]
labels_test = labels[test_idx]


# Check sizes
print("Train:", X_train.shape, "| Calibration:", X_cal.shape, "| Test:", X_test.shape)

Train: (1679285, 64) | Calibration: (419822, 64) | Test: (421549, 64)


### Settings

- **n_estimators = 100**: number of trees; more trees give a smoother average, and past about 100 the gain is small and runs slower.
- **max_features = "sqrt"**: each split only considers about 8 of the 64 features, so trees differ and their mistakes cancel out when averaging.
- **min_samples_leaf = 20**: each leaf must hold at least 20 flows, so trees learn patterns instead of memorizing single flows (same setting as HGB).
- **max_samples = 200,000**: each tree trains on its own 200,000-row bootstrap sample, which keeps memory and time manageable.
- **class_weight = "balanced"**: attack flows count more so benign traffic doesn't dominate (same as HGB).
- **random_state = 5502**: the same seed the team used for HGB.

In [15]:
# Creating the random forest
rf_classifier = RandomForestClassifier(n_estimators=100,
                                       max_features="sqrt",
                                       min_samples_leaf=20,
                                       max_samples=200000,
                                       class_weight="balanced",
                                       n_jobs=-1,
                                       random_state=5502)


### Fitting the forest

The 100 trees are built using only the training rows, so the test day has no influence on what the model learns.

In [16]:
# Fitting Forest on the training data
rf_classifier.fit(X_train, y_train)

,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",5502
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"max_samples max_samples: int or float, default=NoneIf bootstrap is True, the number of samples to draw from Xto train each base estimator.- If None (default), then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples... versionadded:: 0.22.. versionchanged:: 1.9 Float `max_samples` is relative to `sample_weight.sum()` instead of `X.shape[0]` for weighted samples.",200000
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)

### Getting Attack Probabilities 


`predict_proba` gives the share of trees that voted attack for each flow. Column 1 is the attack column.

In [18]:
# Get attack probability for the test day
test_scores = rf_classifier.predict_proba(X_test)[:,1]
test_scores

array([0.01633531, 0.0250584 , 0.01722638, ..., 0.05455424, 0.00247381,
       0.        ], shape=(421549,))

### Setting the alert threshold

The cutoff is set so that only 1% of benign calibration flows would raise an alert, which matches the same rule used for logistic regression and HGB.

In [19]:
# Keep only benign calibration rows
X_cal_benign = X_cal[y_cal == 0]

# Get attack probability for benign calibration rows
cal_scores = rf_classifier.predict_proba(X_cal_benign)[:, 1]

# Set the threshold
threshold = pick_threshold(cal_scores)
threshold

0.16817448859429415

In [20]:
# Turn Scores into predictions using the threshold
y_pred = test_scores > threshold
y_pred

array([False, False, False, ..., False, False, False], shape=(421549,))

### Evaluating the test day

The forest is scored with the team's `evaluate` function, the same metrics used for HGB. Accuracy is not used because always predicting benign already scores over 80%.

- **Recall**: Share of the test day's attacks caught
- **False Alarm Rate**: Share of normal traffic flagged (around 1%)
- **Precision**: Share of alerts that were real attacks
- **PR-AUC**: How well attacks are ranked above normal traffic

In [22]:
# Get Metrics from Evaluate
metrics, per_class = evaluate(y_test, test_scores, threshold, labels_test)

# Show main metrics
print("Recall:", round(metrics["recall"], 3))
print("False alarm rate:", round(metrics["fpr"], 4))
print("Precision:", round(metrics["precision"], 3))
print("PR-AUC:", round(metrics["pr_auc"], 3))

Recall: 0.009
False alarm rate: 0.0279
Precision: 0.007
PR-AUC: 0.059


In [24]:
# Show confusion matrix counts
print("TP:", metrics["tp"], "| FN:", metrics["fn"])
print("FP:", metrics["fp"], "| TN:", metrics["tn"])

TP: 79 | FN: 9071
FP: 11523 | TN: 400876


In [25]:
# Show Recall for each attack type
per_class

{'FTP-Patator': {'n': 5931, 'recall': 0.007587253414264037},
 'SSH-Patator': {'n': 3219, 'recall': 0.01056228642435539}}

So on Tuesday, the forest caught 0.9% of attacks with a false alarm rate of 2.8%. HGB caught 2.6% at 2.6% false alarms. FTP-Patator recall was 0.8% and SSH-Patator recall was 1.1%.

### Why not GridSearchCV

GridSearchCV uses random cross-validation, which would mix flows from the same day into training and testing. That is leakage, because the model would see the attack types it is tested on. It would also take far too long on our dataset which has 2.5 million rows. So the forest is not tuned, which slightly favors the tuned HGB.

### Setting up the leave-one-day-out runs

Each day takes a turn as the unseen test day, including Monday. Monday has no attacks, so it cannot measure detection, but it gives the cleanest test of false alarms. Every day runs with seed 5502 so we can compare with logistic regression and HGB. Tuesday also runs with seeds 1, 2, and 3, the same seeds as `m3_stability.csv`, because Tuesday is  where HGB was unstable. That gives 8 runs in total.

In [26]:
# Days to hold out
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

# Set empty list for runs
RUNS = []

# One seed for every day
for d in DAYS:
    RUNS.append((d, 5502))

# Three extra seeds for Tuesday only
for s in [1, 2, 3]:
    RUNS.append(("Tuesday", s))

# Check the run list
print(len(RUNS), "runs:", RUNS)

8 runs: [('Monday', 5502), ('Tuesday', 5502), ('Wednesday', 5502), ('Thursday', 5502), ('Friday', 5502), ('Tuesday', 1), ('Tuesday', 2), ('Tuesday', 3)]


### Running every day and seed

This loop repeats the Part 1 steps for each run, so every run follows the same rules. Only the test day and seed change. Results are saved to `results/m3/rq4_forest.csv` after every run, so nothing is lost if the kernel stops.

In [28]:
# Set empty list for results
results = []

for test_day, seed in RUNS:
    # Get row numbers for training days and test day
    train_idx = np.flatnonzero(day != test_day)
    test_idx = np.flatnonzero(day == test_day)

    # Split training rows into fit and calibration
    fit_idx, cal_idx = split_fit_calib(train_idx, labels)

    # Set up train, calibration and test data
    X_train = X[fit_idx]
    y_train = y[fit_idx]
    X_cal = X[cal_idx]
    y_cal = y[cal_idx]
    X_test = X[test_idx]
    y_test = y[test_idx]
    labels_test = labels[test_idx]

    # Create and fit the random forest
    rf_classifier = RandomForestClassifier(n_estimators=100,
                                           max_features="sqrt",
                                           min_samples_leaf=20,
                                           max_samples=200_000,
                                           class_weight="balanced",
                                           n_jobs=-1,
                                           random_state=seed)
    rf_classifier.fit(X_train, y_train)

    # Get attack probability for the test day
    test_scores = rf_classifier.predict_proba(X_test)[:, 1]

    # Set the threshold from benign calibration rows
    X_cal_benign = X_cal[y_cal == 0]
    cal_scores = rf_classifier.predict_proba(X_cal_benign)[:, 1]
    threshold = pick_threshold(cal_scores)

    # Get metrics from evaluate
    metrics, per_class = evaluate(y_test, test_scores, threshold, labels_test)

    # Save this run's results
    row = {"day": test_day, "seed": seed,
           "recall": metrics["recall"],
           "fpr": metrics["fpr"],
           "precision": metrics["precision"],
           "pr_auc": metrics["pr_auc"],
           "tp": metrics["tp"], "fn": metrics["fn"],
           "fp": metrics["fp"], "tn": metrics["tn"]}

    # Add recall for each attack type
    for attack in per_class:
        row["recall " + attack] = per_class[attack]["recall"]

    results.append(row)

    # Print progress
    print(test_day, "seed", seed, "| recall:", round(metrics["recall"], 3),
          "| false alarms:", round(metrics["fpr"], 4))

    # Save after every run
    pd.DataFrame(results).to_csv(OUT_DIR / "rq4_forest.csv", index=False)

# Done
print("Done")

Monday seed 5502 | recall: nan | false alarms: 0.0188
Tuesday seed 5502 | recall: 0.009 | false alarms: 0.0279
Wednesday seed 5502 | recall: 0.789 | false alarms: 0.012
Thursday seed 5502 | recall: 0.026 | false alarms: 0.1067
Friday seed 5502 | recall: 0.376 | false alarms: 0.0217
Tuesday seed 1 | recall: 0.099 | false alarms: 0.0275
Tuesday seed 2 | recall: 0.024 | false alarms: 0.0274
Tuesday seed 3 | recall: 0.175 | false alarms: 0.0281
Done


From the four Tuesday seeds, the Forest's recall was 0.009, 0.099, 0.024, and 0.175, so the recall score still changes a lot depending only on the seed, while false alarms stayed near constant at a range around 2.7-2.8%. 

For the other days, the forest caught 78.9% of Wednesday's attacks, 2.6% on Thursday, and 37.6% on Friday.

### Comparing Tuesday across the same four seeds

This is the main test. HGB's Tuesday recall ranged from 0.019 to 0.222 across seeds 5502, 1, 2 and 3. If the forest's range is much smaller, bagging reduced the variance.

In [31]:
# Load forest and HGB seed results
forest = pd.read_csv(OUT_DIR / "rq4_forest.csv")
hgb_seeds = pd.read_csv(OUT_DIR / "m3_stability.csv")

# Get Tuesday rows for each model
forest_tue = forest[forest["day"] == "Tuesday"]
hgb_tue = hgb_seeds[hgb_seeds["day"] == "Tuesday"]

# Set empty list for comparison rows
tue_table = []

# Build min and max for each model
for name, data in [("HGB", hgb_tue), ("Random Forest", forest_tue)]:
    tue_table.append({"model": name,
                      "recall min": data["recall"].min(),
                      "recall max": data["recall"].max(),
                      "recall range": data["recall"].max() - data["recall"].min(),
                      "pr_auc min": data["pr_auc"].min(),
                      "pr_auc max": data["pr_auc"].max()})

# Show Tuesday comparison
pd.DataFrame(tue_table).round(3)

,model,recall min,recall max,recall range,pr_auc min,pr_auc max
0,HGB,0.019,0.222,0.202,0.063,0.109
1,Random Forest,0.009,0.175,0.167,0.059,0.077


Comparing these results we can see that Random Forest's Tuesday recall ranged from 0.009 to 0.175 across the four seeds, compared to HGB's 0.019 to 0.222. The range for the Random Forest is only slightly smaller, so bagging didn't meaningfully reduce the variance on Tuesday. This matches our stability finding that detection of unseen attacks on Tuesday is fragile, but this comparison alone does not show what causes the instability.

### Comparing all three models on every held-out day

This uses the forest's seed 5502 run against the logistic regression and HGB results from `m3_summary_rq4_by_day.csv`.

In [32]:
# Load the RQ4 results for logreg and HGB
rq4 = pd.read_csv(OUT_DIR / "m3_summary_rq4_by_day.csv")
rq4 = rq4[rq4["model"].isin(["logreg", "hgb"])]

# Keep the forest's seed 5502 run for each day
forest_main = forest[forest["seed"] == 5502].copy()
forest_main["model"] = "forest"

# Stack all three models
cols = ["day", "model", "recall", "fpr", "pr_auc"]
compare = pd.concat([rq4[cols], forest_main[cols]])

# Show one table per metric with days as rows and models as columns
for metric in ["pr_auc", "recall", "fpr"]:
    print("\n" + metric)
    print(compare.pivot(index="day", columns="model", values=metric).reindex(DAYS).round(3))


pr_auc
model      forest    hgb  logreg
day                             
Monday        NaN    NaN     NaN
Tuesday     0.059  0.063   0.020
Wednesday   0.954  0.756   0.467
Thursday    0.023  0.031   0.018
Friday      0.808  0.846   0.776

recall
model      forest    hgb  logreg
day                             
Monday        NaN    NaN     NaN
Tuesday     0.009  0.026   0.002
Wednesday   0.789  0.614   0.058
Thursday    0.026  0.912   0.000
Friday      0.376  0.372   0.375

fpr
model      forest    hgb  logreg
day                             
Monday      0.019  0.014   0.016
Tuesday     0.028  0.026   0.015
Wednesday   0.012  0.012   0.014
Thursday    0.107  0.108   0.091
Friday      0.022  0.021   0.014


The PR-AUC results demonstrate that the Random Forest performed better than HGB on Wednesday (0.954 vs 0.756), was slightly lower on Friday (0.808 vs 0.846), and was about the same on Tuesday and Thursday, where both models scored very low and only a little above random guessing. When averaging the four attack days, the Random Forest had a PR-AUC of 0.461, compared with 0.424 for HGB and 0.320 for logistic regression. The Wednesday improvement is likely real and not just from the seed, since 0.954 is higher than HGB's best Wednesday PR-AUC across all four of its seeds (0.897). However, the Random Forest was only run with one seed on Wednesday, so we can't come to a conclusion that it is better.

The recall results show that the Random Forest caught more of Wednesday's attacks (78.9% vs 61.4%), about the same on Friday, and far fewer on Thursday (2.6% vs 91.2%). Thursday is difficult to judge for either model, since PR-AUC was very low for both, false alarms were around 11%, and the team found that most of HGB's Thursday catches were zero-payload web-attack flows that most likely don't contain an attack.

The false alarm rates were almost the same for the Random Forest and HGB on every day except Monday (1.9% vs 1.4%), and none of the models stayed within the 1% budget on any unseen day.

### Conclusion

For our research question 4, we wanted to know how well a model detects attacks on a day it has never seen. The Random Forest model was included since HGB's Tuesday recall jumped from 1.9% to 22% just from changing the random seed, and the bagging method is a common way to reduce variance.

The results from our Random Forest model demonstrated that it did not meaningfully reduce this variance. The Tuesday recall from the Random Forest model ranged from 0.9% to 17.5% across the same four seeds, so a single Tuesday run for either model is not a reliable estimate.

Across all of the days, the Random Forest was competitive with HGB. It ranked attacks much better on Wednesday (PR-AUC 0.954 vs 0.756), was slightly lower on Friday (0.808 vs 0.846), and was about the same on Tuesday and Thursday, where both models scored very low. It caught more of Wednesday's attacks (78.9% vs 61.4%) but far fewer of Thursday's (2.6% vs 91.2%). However, Thursday is hard to judge, since most of HGB's catches there were zero-payload web-attack flows that most likely don't contain an attack, so HGB's 91.2% is likely higher than its true detection rate. False alarms were about the same for both models, and neither stayed within the 1% budget on any unseen day.


Overall, the Random Forest is a reasonable alternative to HGB, especially on Wednesday's DoS attacks, but it doesn't solve the main problem in research question 4, since detection of attacks the model hasn't seen is still not reliable, with Tuesday's and Thursday's attacks being mostly missed, regardless of the ensemble method that is used. Since the Random Forest was not tuned and used only one seed on most days, we keep HGB as the main model and have the Random Forest as a comparison.